In [ ]:
import pandas as pd
import xlwings as xw

from process_engine import f1_confirm_test_file_folder


# ---------------------------------------------------------------------------
# Main Function: process_comms_main
# ---------------------------------------------------------------------------
def process_comms_main():
    """
    Overall Process Commissions Main Function

    Step 1:
        Confirm whether this is:
        - Test run or Official run
        - File or Folder import
        - Whether to clear table

    Step 2:
        Get workbook, output sheet, comm_month, and lookup df
    """

    # -----------------------------------------------------------------------
    # Step 1: Confirm Test / Official / File / Folder / Clear Table
    # -----------------------------------------------------------------------
    print("🟦 Step 1: Confirm Test / Official / File / Folder")
    test_official, file_folder, clear_table = f1_confirm_test_file_folder()
    print("✅ Step 1 complete")
    print(f"   [F1] test_official: {test_official}")
    print(f"   [F1] file_folder   : {file_folder}")
    print(f"   [F1] clear_table   : {clear_table}")

    # -----------------------------------------------------------------------
    # Step 2: Get workbook, comm_month, lookup df, and output sheet
    # -----------------------------------------------------------------------
    print("🟦 Step 2: Get workbook, comm_month, lookup df, and output sheet")
    current_wb, sh_output, comm_month, comm_tables_main_df = f2_get_workbook_and_vars(
        test_official=test_official
    )
    print("✅ Step 2 complete")
    print(f"   [F2] comm_month: {comm_month}")
    print(
        f"   [F2] comm_tables_main_df shape: "
        f"{comm_tables_main_df.shape if comm_tables_main_df is not None else None}"
    )
    print(f"   [F2] output_sheet: {sh_output.name if sh_output is not None else None}")

    # -----------------------------------------------------------------------
    # Step 3: Placeholder for next logic
    # -----------------------------------------------------------------------
    print("🟨 Step 3: Next logic still to be rebuilt")
    print("   - This is where we will route into:")
    print("     - test single-file logic")
    print("     - official single-file logic")
    print("     - official folder logic")


# ---------------------------------------------------------------------------
# F2: Get Workbook, Sheets, Variables, Lookup DF
# ---------------------------------------------------------------------------
def f2_get_workbook_and_vars(test_official):
    """
    Returns:
      current_wb, sh_output, comm_month, comm_tables_main_df
    """
    current_wb = xw.books.active

    # output sheet
    if str(test_official).strip().lower() == "test":
        out_name = "processed_comms_test"
    else:
        out_name = "processed_comms"

    existing_sheet_names_lower = [s.name.lower() for s in current_wb.sheets]

    if out_name.lower() in existing_sheet_names_lower:
        sh_output = current_wb.sheets[out_name]
    else:
        sh_output = current_wb.sheets.add(out_name)

    # comm_month
    comm_month = None
    try:
        sh_dash = current_wb.sheets["Dashboard"]
        comm_month = sh_dash.range("C3").value
        print("   [F2] Comm Month Read")
        print(f"   [F2] {comm_month}")
    except Exception as e:
        print(f"   [F2] Failed to read comm_month: {e}")
        comm_month = None

    # lookup df
    comm_tables_main_df = pd.DataFrame()
    try:
        sh_lookup = current_wb.sheets["Lookup_Tables"]

        last_row = sh_lookup.range("BC4000").end("up").row
        if last_row < 3:
            last_row = 3

        values = sh_lookup.range(f"BC3:BM{last_row}").value

        expected_headers = [
            "Product House",
            "Contract Number Action", "Contract Number Details",
            "Client Name Action", "Client Name Details",
            "Planner Action", "Planner Details",
            "Total Commission Action", "Total Commission Details",
            "Commission Date Action", "Commission Date Details",
        ]

        comm_tables_main_df = pd.DataFrame(values, columns=expected_headers)
        comm_tables_main_df = comm_tables_main_df.dropna(how="all")

    except Exception as e:
        print(f"   [F2] Failed to read comm_tables_main_df: {e}")
        comm_tables_main_df = pd.DataFrame()

    return current_wb, sh_output, comm_month, comm_tables_main_df

In [ ]:
process_comms_main()